# 轨迹矩阵居中裁切工具

将已提取的 100×100 轨迹矩阵居中裁切为指定的正方形尺寸。

## 流程
1. 配置路径与目标裁切尺寸
2. 干跑检查：扫描所有文件，检测是否有轨迹超出裁切范围
3. 确认无问题后，执行正式裁切并保存

## 1. 路径与参数配置

In [ ]:
from pathlib import Path
import numpy as np
from typing import List, Dict

# TODO: 修改以下路径
INPUT_DIR = Path(r"d:\Project\Timepix\ProcessData\C2_Original")
OUTPUT_DIR = Path(r"d:\Project\Timepix\ProcessData\C2_Cropped")

CURRENT_SIZE = 100   # 当前矩阵尺寸
CROP_SIZE = 32       # 目标裁切尺寸
DECIMAL_PRECISION = 6

print(f"输入目录: {INPUT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"裁切: {CURRENT_SIZE}×{CURRENT_SIZE} → {CROP_SIZE}×{CROP_SIZE}")

## 2. 工具函数

In [ ]:
def list_txt_files(input_dir: Path) -> List[Path]:
    """递归列出目录下所有 txt 文件。"""
    return sorted(input_dir.rglob("*.txt"))


def get_crop_slice(current_size: int, crop_size: int) -> slice:
    """计算居中裁切的切片范围。"""
    offset = (current_size - crop_size) // 2
    return slice(offset, offset + crop_size)


def has_active_pixels_outside_crop(matrix: np.ndarray, crop_slc: slice) -> bool:
    """检查裁切区域外是否存在激活像素（非零值）。"""
    size = matrix.shape[0]
    mask = np.zeros_like(matrix, dtype=bool)
    mask[crop_slc, crop_slc] = True
    outside = matrix[~mask]
    return np.any(outside > 0)

## 3. 干跑检查

扫描所有文件，检测是否有轨迹的激活像素超出裁切范围。如果有，列出文件名；如果没有，提示可以安全裁切。

In [ ]:
txt_files = list_txt_files(INPUT_DIR)
crop_slc = get_crop_slice(CURRENT_SIZE, CROP_SIZE)

print(f"找到 {len(txt_files)} 个文件")
print(f"裁切范围: [{crop_slc.start}:{crop_slc.stop}, {crop_slc.start}:{crop_slc.stop}]")
print(f"正在检查...\n")

overflow_files = []

for file_path in txt_files:
    matrix = np.loadtxt(file_path, dtype=np.float64)
    if matrix.shape != (CURRENT_SIZE, CURRENT_SIZE):
        print(f"跳过尺寸不匹配的文件: {file_path.name} ({matrix.shape})")
        continue
    if has_active_pixels_outside_crop(matrix, crop_slc):
        rel = file_path.relative_to(INPUT_DIR)
        overflow_files.append(rel)

print("=" * 50)
if overflow_files:
    print(f"⚠ 发现 {len(overflow_files)} 个文件的轨迹超出 {CROP_SIZE}×{CROP_SIZE} 裁切范围:\n")
    for f in overflow_files:
        print(f"  {f}")
    print(f"\n这些文件在裁切后会丢失部分激活像素。")
else:
    print(f"✓ 所有 {len(txt_files)} 个文件均可安全裁切为 {CROP_SIZE}×{CROP_SIZE}，无信息丢失。")

## 4. 执行裁切

确认干跑检查通过后，运行以下单元格执行正式裁切并保存。

In [ ]:
crop_slc = get_crop_slice(CURRENT_SIZE, CROP_SIZE)
saved = 0
skipped = 0

for file_path in txt_files:
    matrix = np.loadtxt(file_path, dtype=np.float64)
    if matrix.shape != (CURRENT_SIZE, CURRENT_SIZE):
        skipped += 1
        continue

    cropped = matrix[crop_slc, crop_slc]

    rel_path = file_path.relative_to(INPUT_DIR)
    output_path = OUTPUT_DIR / rel_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(output_path, cropped, fmt=f"%.{DECIMAL_PRECISION}f")
    saved += 1

print(f"裁切完成: 保存 {saved} 个文件，跳过 {skipped} 个文件")
print(f"输出目录: {OUTPUT_DIR}")

## 5. 可视化对比（可选）

随机抽取样本，对比裁切前后效果。

In [ ]:
import matplotlib.pyplot as plt
import random

sample_files = random.sample(txt_files, min(4, len(txt_files)))

fig, axes = plt.subplots(len(sample_files), 2, figsize=(8, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = [axes]

for row, file_path in zip(axes, sample_files):
    original = np.loadtxt(file_path, dtype=np.float64)

    rel_path = file_path.relative_to(INPUT_DIR)
    cropped_path = OUTPUT_DIR / rel_path
    cropped = np.loadtxt(cropped_path, dtype=np.float64)

    row[0].imshow(original, cmap='hot', interpolation='nearest')
    row[0].set_title(f"原始 {CURRENT_SIZE}×{CURRENT_SIZE}", fontsize=9)
    row[0].axis('off')

    row[1].imshow(cropped, cmap='hot', interpolation='nearest')
    row[1].set_title(f"裁切 {CROP_SIZE}×{CROP_SIZE}", fontsize=9)
    row[1].axis('off')

plt.tight_layout()
plt.show()